# Text Extraction with OCR: A Step-by-Step Practical Course

Welcome! This notebook is a hands-on mini course to help you understand and practice OCR (Optical Character Recognition) for text extraction.

## Learning Goals
By the end of this notebook, you will be able to:
1. Explain what OCR is and where it is used.
2. Understand different OCR approaches and tools.
3. Run OCR with **Tesseract** and **EasyOCR**.
4. Try an additional **Deep Learning OCR** example with **TrOCR**.
5. Improve OCR results using image preprocessing.
6. Compare outputs from different OCR techniques.
7. Build a small reusable OCR pipeline.
8. Call a **Cloud OCR API** with a free demo mode (no manual key registration).
9. **Visualize** OCR output with bounding boxes and labels on input images.

## Course Structure
- Part 1: OCR Concepts and Types
- Part 2: Environment Setup
- (Early Part 3) OCR visualization helpers — boxes + labels on images
- Part 3: Read PDF and convert pages to images
- Part 4: OCR with Tesseract
- Part 5: OCR with EasyOCR (full-page overlays + optional tiling)
- Part 5.1a–c (Optional): Tile split / per-tile EasyOCR / stitch + overlay (same pattern as Part 3.1)
- Part 5.2 (Optional): Deep Learning OCR with TrOCR
- Part 6: Preprocessing Techniques
- Part 7: Side-by-side Comparison
- Part 8: Mini Pipeline
- Part 9: Cloud OCR API (free demo mode)
- Final Notes + Exercises


## Part 1 - OCR Basics and Types

### What is OCR?
OCR (Optical Character Recognition) converts text in images (photos, scanned documents, screenshots) into machine-readable text.

### Common OCR use cases
- Invoice and receipt processing
- ID/passport and form digitization
- License plate recognition
- Archive digitization (books/newspapers)
- Accessibility (reading text from images)

### Main OCR approaches
1. **Traditional OCR**
   - Uses image processing + pattern matching + language models.
   - Example: **Tesseract**.
   - Fast and lightweight, but sensitive to image quality.

2. **Deep Learning OCR**
   - Uses neural networks for text detection and recognition.
   - Example: **EasyOCR** (uses deep models under the hood).
   - Usually better on difficult images but heavier.

3. **Cloud OCR APIs**
   - Managed services from cloud providers.
   - Very strong performance and multilingual support.
   - Requires internet, API keys, and often paid usage.

In this lesson, we focus on local OCR with **Tesseract** and **EasyOCR**.

## Part 2 - Environment Setup (Run this first)

This cell installs Python libraries used in this notebook.

> Note:
> - `pytesseract` is a Python wrapper, but you also need the **Tesseract executable** installed on your OS.
> - On Windows, install from UB Mannheim or official packages and update `tesseract_cmd` path below if needed.


In [ ]:
!python.exe -m pip install --upgrade pip

In [ ]:
%pip install -q opencv-python pillow pytesseract easyocr matplotlib numpy requests pymupdf --user

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pytesseract
import easyocr
import requests
import fitz  # PyMuPDF for PDF -> image conversion

print('Imports completed successfully.')

### Configure Tesseract path (important on Windows)

If you get an error like `TesseractNotFoundError`, set the full path to `tesseract.exe`.
Common example path:
`C:\\Program Files\\Tesseract-OCR\\tesseract.exe`

In [ ]:
# Uncomment and edit if needed:
# pytesseract.pytesseract.tesseract_cmd = r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"

try:
    version = pytesseract.get_tesseract_version()
    print('Tesseract detected:', version)
except Exception as e:
    print('Tesseract not detected yet. Please install and configure path.')
    print('Error:', e)

## Part 3 - Read PDF and Convert to Images for OCR

In real OCR projects, input is often a **PDF** instead of a pre-made image.
To make learning easier, this part is split into small runnable steps.

### Step overview
1. Set and validate PDF path.
2. Convert PDF pages to images.
3. Visualize page images.
4. Apply basic preprocessing.
5. **Full-page OCR** (before any tiling): extract text and **draw word boxes + labels** on the page image.
6. (Optional) **Part 3.1**: split the page into a **controlled grid** of tiles, OCR each tile, merge text, stitch — use **the same tiles** from 3.1a in 3.1b (no second split).

This section prepares image data that can be passed to Tesseract, EasyOCR, and Cloud OCR APIs.


In [ ]:
# Step 1) Set and validate the input PDF path.
os.makedirs('data', exist_ok=True)
pdf_path = 'data/Sample-DWG1.pdf'

if not os.path.exists(pdf_path):
    raise FileNotFoundError(
        f"PDF not found at {pdf_path}. Place a PDF there, then re-run this cell."
    )

print('PDF path is valid:', pdf_path)

In [ ]:
# Step 2) Define helper functions for conversion and visualization.

def pdf_to_images(input_pdf_path, output_dir='data/pdf_pages', zoom=2.0):
    """
    Convert each PDF page to a PNG image.

    zoom: controls render resolution.
      - Higher zoom => sharper text for OCR
      - But also larger file size and more memory/time
    """
    os.makedirs(output_dir, exist_ok=True)

    doc = fitz.open(input_pdf_path)
    image_paths = []

    for page_idx in range(len(doc)):
        page = doc[page_idx]
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)

        out_path = os.path.join(output_dir, f'page_{page_idx + 1}.png')
        pix.save(out_path)
        image_paths.append(out_path)

    doc.close()
    return image_paths


def show_image(path, title='Image'):
    img_bgr = cv2.imread(path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 6))
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

### Step 2 - Convert PDF pages to images

Run this cell to render every page in the PDF into PNG files.

Notes:
- `zoom=2.0` is a good default for OCR readability.
- If OCR misses small text, increase `zoom` (for example, 2.5 or 3.0).
- Higher zoom also increases processing time and memory usage.

In [ ]:
page_image_paths = pdf_to_images(pdf_path, zoom=2.0)
print(f'Converted {len(page_image_paths)} pages from PDF to images.')
for p in page_image_paths:
    print('-', p)

# Keep variable names used in later OCR parts.
clean_path = page_image_paths[0]
print('Main OCR page:', clean_path)

### Step 3 - Visualize page images and create an optional stress-test image

This step helps learners inspect input quality before OCR.

We also generate a noisy + slightly rotated copy (`noisy_path`) to simulate difficult real-world scans.
That allows direct comparison between easy and hard OCR inputs in later parts.

In [ ]:
show_image(clean_path, 'PDF Page 1 (Rendered Image)')

# Optional degraded version for robustness tests (learning purpose only)
base_img = Image.open(clean_path).convert('RGB')
arr = np.array(base_img)
noise = np.random.normal(0, 20, arr.shape).astype(np.int16)
noisy = np.clip(arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
noisy_img = Image.fromarray(noisy).rotate(1.5, expand=True, fillcolor='white')
noisy_path = 'data/page1_noisy_rotated.png'
noisy_img.save(noisy_path)

show_image(noisy_path, 'Optional Degraded Version for OCR Stress Test')
print('Optional degraded test page:', noisy_path)

### Step 4 - Apply preprocessing before OCR

Preprocessing usually improves OCR accuracy, especially on noisy scans.

In this step:
- Convert to grayscale to simplify data
- Denoise to suppress random artifacts
- Apply adaptive thresholding to strengthen text/background contrast

Output file from this step (`preprocessed_pdf_path`) can be used directly for OCR.

In [ ]:
def preprocess_pdf_page(image_path, save_path='data/page_preprocessed.png'):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Bilateral filter keeps edges while reducing noise.
    denoised = cv2.bilateralFilter(gray, 9, 75, 75)

    # Adaptive threshold works better when lighting is uneven.
    bw = cv2.adaptiveThreshold(
        denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 15
    )

    cv2.imwrite(save_path, bw)
    return save_path

preprocessed_pdf_path = preprocess_pdf_page(clean_path)
show_image(preprocessed_pdf_path, 'Preprocessed PDF Page 1')

### Step 5 — Full-page OCR before splitting (text + boxes on image)

Run **Tesseract** on the **full rendered page** (`clean_path`) — still **one image**, no tiles yet.

We print the extracted string and save a visualization with **word-level boxes** and **labels** (using `pytesseract.image_to_data`).

This answers: *what does OCR see on the whole page before we cut it into tiles?*


In [ ]:
def draw_tesseract_word_boxes_on_image(image_path, lang='eng', psm=6, max_label_len=50):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)
    config = f'--psm {psm}'
    data = pytesseract.image_to_data(
        Image.open(image_path), lang=lang, config=config,
        output_type=pytesseract.Output.DICT,
    )
    n = len(data.get('text', []))
    for j in range(n):
        try:
            conf = int(float(data['conf'][j]))
        except (ValueError, TypeError):
            conf = -1
        if conf < 0:
            continue
        t = (data['text'][j] or '').strip()
        if not t:
            continue
        x, y, ww, hh = data['left'][j], data['top'][j], data['width'][j], data['height'][j]
        if ww <= 0 or hh <= 0:
            continue
        cv2.rectangle(img, (x, y), (x + ww, y + hh), (0, 160, 255), 2)
        lab = (t[:max_label_len] + '...') if len(t) > max_label_len else t
        cv2.putText(
            img, lab, (x, max(14, y - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 0, 120), 1, cv2.LINE_AA,
        )
    return img


print('Step 5 | Full-page OCR (clean_path, no tiles yet)\n')
step5_text = pytesseract.image_to_string(Image.open(clean_path), lang='eng', config='--psm 6')
print(step5_text)

step5_vis = draw_tesseract_word_boxes_on_image(clean_path, lang='eng', psm=6)
step5_out = 'data/part3_step5_fullpage_ocr_overlay.png'
cv2.imwrite(step5_out, step5_vis)
show_image(step5_out, 'Step 5 — Full-page OCR: word boxes + labels on input image')


### Part 3.1 (Optional) Split Very Large Images Before OCR

Why do we need to split a large page image?

1. **Memory and speed**: Very high-resolution page images can be expensive to process.
2. **Recognition stability**: Some OCR engines perform better on smaller regions than on one huge full-page image.
3. **Small text recovery**: Tiling can preserve detail for tiny fonts when full-page OCR misses them.
4. **Failure isolation**: If one tile fails, other tiles can still be processed.

Trade-off:
- If tiles are too small, reading order and context can be harder to reconstruct.
- We use a small `overlap` to reduce text truncation near tile borders.

Use this step only when page size is large or OCR quality is unstable on full-page input.

### Part 3.1 outline (run cells below in order)

This matches **Step 6** in the Part 3 overview:

| Section | What you do |
|--------|-------------|
| **3.1a** | Split the page into tiles, save crops, draw **tile boundaries** on the original image |
| **3.1b** | Run **OCR on each tile** (Tesseract via `pytesseract`), then **merge text** in reading order |
| **3.1c** | **Stitch** tile images back onto a full-size canvas and **compare** with the original |

> `pytesseract` is used here so tile OCR works **before** Part 4 defines `ocr_tesseract`.




In [ ]:
def split_image_grid(image_path, n_rows, n_cols, output_dir='data/tiles', overlap=0):
    """
    Split the page into exactly n_rows * n_cols tiles (uniform grid).
    Adjust n_rows and n_cols to avoid too many tiles (e.g. 2x2=4, 3x2=6).
    overlap: optional extra pixels; keep 0 unless you know you need it.
    """
    os.makedirs(output_dir, exist_ok=True)
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')
    h, w = img.shape[:2]
    records = []
    row_edges = [int(round(i * h / n_rows)) for i in range(n_rows + 1)]
    col_edges = [int(round(j * w / n_cols)) for j in range(n_cols + 1)]
    tile_id = 0
    for ri in range(n_rows):
        for ci in range(n_cols):
            y1, y2 = row_edges[ri], row_edges[ri + 1]
            x1, x2 = col_edges[ci], col_edges[ci + 1]
            if overlap > 0:
                y1 = max(0, y1 - overlap // 2)
                x1 = max(0, x1 - overlap // 2)
                y2 = min(h, y2 + overlap // 2)
                x2 = min(w, x2 + overlap // 2)
            tile = img[y1:y2, x1:x2]
            tile_path = os.path.join(output_dir, f'tile_grid_{tile_id}.png')
            cv2.imwrite(tile_path, tile)
            records.append({
                'path': tile_path,
                'y': y1, 'x': x1, 'y2': y2, 'x2': x2,
                'tile_id': tile_id,
                'row': ri, 'col': ci,
            })
            tile_id += 1
    return records, (h, w)


def split_large_image(image_path, output_dir='data/tiles', max_side=1800, overlap=80):
    'Optional: split by max tile size + overlap (can create many tiles on large pages).'
    os.makedirs(output_dir, exist_ok=True)
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f'Could not read image: {image_path}')
    h, w = img.shape[:2]
    records = []
    if max(h, w) <= max_side:
        records.append({'path': image_path, 'y': 0, 'x': 0, 'y2': h, 'x2': w, 'tile_id': 0, 'row': 0, 'col': 0})
        return records, (h, w)
    step = max_side - overlap
    tile_id = 0
    for y in range(0, h, step):
        for x in range(0, w, step):
            y2 = min(y + max_side, h)
            x2 = min(x + max_side, w)
            tile = img[y:y2, x:x2]
            tile_path = os.path.join(output_dir, f'tile_{tile_id}.png')
            cv2.imwrite(tile_path, tile)
            records.append({'path': tile_path, 'y': y, 'x': x, 'y2': y2, 'x2': x2, 'tile_id': tile_id, 'row': None, 'col': None})
            tile_id += 1
    return records, (h, w)


def stitch_tiles_to_canvas(records, shape_hw):
    h, w = shape_hw
    canvas = np.full((h, w, 3), 255, dtype=np.uint8)
    for r in sorted(records, key=lambda z: (z['y'], z['x'])):
        tile = cv2.imread(r['path'])
        if tile is None:
            continue
        y, x = r['y'], r['x']
        th, tw = tile.shape[:2]
        canvas[y:y + th, x:x + tw] = tile
    return canvas


def draw_tile_boxes_on_image(image_path, records):
    img = cv2.imread(image_path).copy()
    for r in records:
        cv2.rectangle(img, (r['x'], r['y']), (r['x2'] - 1, r['y2'] - 1), (0, 200, 0), 3)
        cv2.putText(img, str(r['tile_id']), (r['x'] + 6, r['y'] + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 200, 0), 2)
    return img


# --- Part 3.1a: choose grid size (rows x cols). Keep total tiles small for class demos.
N_ROWS = 2
N_COLS = 2
TILE_OVERLAP = 0  # set > 0 only if you need overlap between adjacent tiles

tile_records, page_hw = split_image_grid(
    clean_path,
    n_rows=N_ROWS,
    n_cols=N_COLS,
    output_dir='data/tiles_31a',
    overlap=TILE_OVERLAP,
)
print(f'3.1a | Grid {N_ROWS}x{N_COLS} => {len(tile_records)} tiles  |  Page (H,W): {page_hw}')

boxed = draw_tile_boxes_on_image(clean_path, tile_records)
cv2.imwrite('data/part31a_tile_boxes.png', boxed)
show_image('data/part31a_tile_boxes.png', 'Part 3.1a — Tile grid on original page')

if tile_records:
    show_image(tile_records[0]['path'], 'Part 3.1a — First tile preview')


### Part 3.1b — OCR each tile and merge text

We **reuse the same `tile_records` from Part 3.1a** (no second split). If you change the grid in 3.1a, re-run 3.1a before this cell.

For each tile we run **Tesseract** (`ocr_tile_tesseract`).

**Reading order:** sort by `(y, x)` (row-major).

The tile **gallery** uses `gallery_cols` only for layout (matplotlib), not for splitting.


In [ ]:
def ocr_tile_tesseract(tile_path, lang='eng', psm=6):
    config = f'--psm {psm}'
    return pytesseract.image_to_string(Image.open(tile_path), lang=lang, config=config)


def draw_tesseract_word_boxes_on_tile(image_path, lang='eng', psm=6, max_label_len=40):
    img = cv2.imread(image_path)
    if img is None:
        return None
    config = f'--psm {psm}'
    data = pytesseract.image_to_data(
        Image.open(image_path), lang=lang, config=config,
        output_type=pytesseract.Output.DICT,
    )
    n = len(data.get('text', []))
    for j in range(n):
        try:
            conf = int(float(data['conf'][j]))
        except (ValueError, TypeError):
            conf = -1
        if conf < 0:
            continue
        t = (data['text'][j] or '').strip()
        if not t:
            continue
        x, y, ww, hh = data['left'][j], data['top'][j], data['width'][j], data['height'][j]
        if ww <= 0 or hh <= 0:
            continue
        cv2.rectangle(img, (x, y), (x + ww, y + hh), (0, 180, 0), 1)
        lab = (t[:max_label_len] + '...') if len(t) > max_label_len else t
        cv2.putText(img, lab, (x, max(12, y - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.35, (200, 0, 0), 1, cv2.LINE_AA)
    return img


def show_tile_gallery(records, cols=3, title='Part 3.1b — Tile gallery'):
    n = len(records)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows))
    axes = np.atleast_1d(axes).flatten()
    for i, r in enumerate(records):
        im = cv2.imread(r['path'])
        axes[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f"Tile {r['tile_id']}  (row={r.get('row')}, col={r.get('col')})")
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


# Layout for matplotlib only (how many thumbnails per row in the gallery)
gallery_cols = 3

# Same tiles as 3.1a — do NOT split again
demo_records = tile_records
demo_hw = page_hw

print('3.1b | Using', len(demo_records), 'tiles from Part 3.1a')

show_image(clean_path, 'Part 3.1b — Full page (reference)')
show_tile_gallery(demo_records, cols=gallery_cols)

merged_chunks = []
for r in sorted(demo_records, key=lambda z: (z['y'], z['x'])):
    txt = ocr_tile_tesseract(r['path']).strip()
    merged_chunks.append(
        f"--- Tile {r['tile_id']} (y={r['y']}, x={r['x']}) ---\n{txt}"
    )
merged_ocr_text = '\n\n'.join(merged_chunks)
print('\n' + '=' * 60)
print('MERGED OCR (all tiles, reading order)')
print('=' * 60 + '\n')
print(merged_ocr_text)

# Optional: word boxes on each tile (OCR drawn on tile images)
annotated_records = []
for r in sorted(demo_records, key=lambda z: (z['y'], z['x'])):
    vis = draw_tesseract_word_boxes_on_tile(r['path'], lang='eng', psm=6)
    if vis is None:
        continue
    outp = os.path.join('data', f"tile_annot_{r['tile_id']}.png")
    cv2.imwrite(outp, vis)
    annotated_records.append({**r, 'path': outp})

if annotated_records:
    show_tile_gallery(annotated_records, cols=gallery_cols, title='Part 3.1b — Tiles with Tesseract word boxes + labels')


### Part 3.1c — Stitch tiles back to a full image

We rebuild a **canvas** with the same height and width as the original page, then paste each tile at `(y, x)`.

**Visual check:** side-by-side **original** vs **reconstructed**. They should match closely (overlap zones may look slightly different because of the simple overwrite rule).

We then run **Tesseract word-level OCR** on each tile (same tiles as 3.1a), map boxes to **full-page coordinates**, and draw **all extracted words** (boxes + labels) on the stitched canvas before saving.

Saved file: `data/part31c_stitched.png`.


In [ ]:
# Same tiles as 3.1a / 3.1b — stitch, then draw all per-tile OCR words on the full canvas

def draw_tesseract_words_on_stitched_canvas(canvas_bgr, records, lang='eng', psm=6, max_label_len=50):
    """For each tile, run image_to_data, offset boxes by tile (y,x), draw boxes + labels on full image."""
    out = canvas_bgr.copy()
    config = f'--psm {psm}'
    for r in sorted(records, key=lambda z: (z['y'], z['x'])):
        data = pytesseract.image_to_data(
            Image.open(r['path']), lang=lang, config=config,
            output_type=pytesseract.Output.DICT,
        )
        n = len(data.get('text', []))
        ox, oy = int(r['x']), int(r['y'])
        for j in range(n):
            try:
                conf = int(float(data['conf'][j]))
            except (ValueError, TypeError):
                conf = -1
            if conf < 0:
                continue
            t = (data['text'][j] or '').strip()
            if not t:
                continue
            x, y, ww, hh = data['left'][j], data['top'][j], data['width'][j], data['height'][j]
            if ww <= 0 or hh <= 0:
                continue
            gx, gy = ox + x, oy + y
            cv2.rectangle(out, (gx, gy), (gx + ww, gy + hh), (0, 160, 255), 2)
            lab = (t[:max_label_len] + '...') if len(t) > max_label_len else t
            cv2.putText(
                out, lab, (gx, max(14, gy - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (180, 0, 120), 1, cv2.LINE_AA,
            )
    return out


stitched = stitch_tiles_to_canvas(tile_records, page_hw)
stitched_with_ocr = draw_tesseract_words_on_stitched_canvas(stitched, tile_records)
stitch_path = 'data/part31c_stitched.png'
cv2.imwrite(stitch_path, stitched_with_ocr)

orig_rgb = cv2.cvtColor(cv2.imread(clean_path), cv2.COLOR_BGR2RGB)
stitch_rgb = cv2.cvtColor(stitched_with_ocr, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(orig_rgb)
ax[0].set_title('Original page')
ax[0].axis('off')
ax[1].imshow(stitch_rgb)
ax[1].set_title('Stitched + all extracted text (per-tile OCR, mapped to page)')
ax[1].axis('off')
plt.suptitle('Part 3.1c — Original vs stitched with OCR overlay', y=1.02)
plt.tight_layout()
plt.show()

print('Saved:', stitch_path)


## Part 4 - OCR with Tesseract

Tesseract typically works best when text is clear, high-contrast, and well-aligned.

### Key options
- `lang='eng'`: use English language model.
- `--psm`: page segmentation mode.
  - `6`: Assume a uniform block of text.
  - Other modes may work better depending on your layout.


In [ ]:
def ocr_tesseract(image_path, lang='eng', psm=6):
    config = f'--psm {psm}'
    text = pytesseract.image_to_string(Image.open(image_path), lang=lang, config=config)
    return text

print('Tesseract result (clean image):\n')
print(ocr_tesseract(clean_path))

print('\n' + '='*60 + '\n')
print('Tesseract result (noisy image):\n')
print(ocr_tesseract(noisy_path))

## Part 5 - OCR with EasyOCR

EasyOCR includes deep learning-based text detection and recognition.
It often handles noise, rotation, and mixed layouts better than classic OCR.

First run may be slower because model weights are downloaded.

### What you will do in Part 5
1. Run **EasyOCR** on the **clean** and **noisy** page images and print extracted text.
2. **Redraw** both images with **boxes + labels** (detected text and confidence), then **save** the two overlay PNGs under `data/`.
3. (Optional) **Part 5.1a–c**: same **split → OCR per tile → stitch + full-page overlay** flow as Part 3.1, using **EasyOCR** instead of Tesseract.

> TrOCR moved to **Part 5.2** below so numbering matches the tiling subsections.


In [ ]:
# Initialize once (can take time on first run)
reader = easyocr.Reader(['en'], gpu=False)

def ocr_easyocr(image_path):
    results = reader.readtext(image_path)
    # Each item: [bbox, text, confidence]
    texts = [r[1] for r in results]
    return '\n'.join(texts), results

print('EasyOCR result (clean image):\n')
easy_clean_text, easy_clean_raw = ocr_easyocr(clean_path)
print(easy_clean_text)

print('\n' + '='*60 + '\n')
print('EasyOCR result (noisy image):\n')
easy_noisy_text, easy_noisy_raw = ocr_easyocr(noisy_path)
print(easy_noisy_text)

In [ ]:
def draw_easyocr_overlay_bgr(image_path, results, max_label_len=42):
    """Return a BGR image copy with polylines + text labels for each EasyOCR detection."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)
    out = img.copy()
    for bbox, text, conf in results:
        pts = np.array(bbox, dtype=np.int32)
        cv2.polylines(out, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
        x, y = int(pts[0][0]), int(pts[0][1])
        t = (text or '').strip()
        lab = (t[:max_label_len] + '...') if len(t) > max_label_len else t
        label = f'{lab} ({float(conf):.2f})'
        cv2.putText(
            out, label, (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
            (255, 0, 0), 1, cv2.LINE_AA,
        )
    return out


part5_clean_overlay_path = 'data/part5_easyocr_clean_overlay.png'
part5_noisy_overlay_path = 'data/part5_easyocr_noisy_overlay.png'

easy_clean_vis = draw_easyocr_overlay_bgr(clean_path, easy_clean_raw)
easy_noisy_vis = draw_easyocr_overlay_bgr(noisy_path, easy_noisy_raw)
cv2.imwrite(part5_clean_overlay_path, easy_clean_vis)
cv2.imwrite(part5_noisy_overlay_path, easy_noisy_vis)

show_image(part5_clean_overlay_path, 'Part 5 — EasyOCR on clean image (boxes + extracted text)')
show_image(part5_noisy_overlay_path, 'Part 5 — EasyOCR on noisy image (boxes + extracted text)')
print('Saved:', part5_clean_overlay_path)
print('Saved:', part5_noisy_overlay_path)

### Part 5.1 — Optional tiling with EasyOCR (mirrors Part 3.1)

Same **grid split → per-tile OCR → stitch + draw all detections on the full canvas** as **Part 3.1a / 3.1b / 3.1c**, but OCR uses **EasyOCR** (`reader.readtext`).

**Prerequisite:** run the **Part 3.1a** code cell so `split_image_grid`, `stitch_tiles_to_canvas`, and `draw_tile_boxes_on_image` exist. Run **Part 5** cells above so `reader` and `draw_easyocr_overlay_bgr` exist.

| Section | What you do |
|--------|-------------|
| **5.1a** | Set `EZ_N_ROWS` × `EZ_N_COLS`, split `clean_path`, save tiles under `data/tiles_51a_easy/`, draw tile boundaries |
| **5.1b** | Reuse `ez_tile_records` from 5.1a (**no second split**), EasyOCR each tile + merge text, optional per-tile overlay gallery |
| **5.1c** | Stitch tiles, draw every EasyOCR detection in **page coordinates**, save `data/part51c_easyocr_stitched_overlay.png` |


In [ ]:
if 'split_image_grid' not in globals():
    raise NameError(
        'Run the Part 3.1a code cell first (defines split_image_grid, stitch_tiles_to_canvas, draw_tile_boxes_on_image).'
    )

# Part 5.1a — same grid idea as Part 3.1a; change rows/cols to control tile count
EZ_N_ROWS = 2
EZ_N_COLS = 2
EZ_TILE_OVERLAP = 0

ez_tile_records, ez_page_hw = split_image_grid(
    clean_path,
    n_rows=EZ_N_ROWS,
    n_cols=EZ_N_COLS,
    output_dir='data/tiles_51a_easy',
    overlap=EZ_TILE_OVERLAP,
)
print(f'5.1a | EasyOCR tiling grid {EZ_N_ROWS}x{EZ_N_COLS} => {len(ez_tile_records)} tiles  |  Page (H,W): {ez_page_hw}')

boxed_ez = draw_tile_boxes_on_image(clean_path, ez_tile_records)
part51a_boxes_path = 'data/part51a_easyocr_tile_boxes.png'
cv2.imwrite(part51a_boxes_path, boxed_ez)
show_image(part51a_boxes_path, 'Part 5.1a — Tile grid on page (EasyOCR pipeline)')

if ez_tile_records:
    show_image(ez_tile_records[0]['path'], 'Part 5.1a — First tile preview')


### Part 5.1b — EasyOCR each tile and merge text

We **reuse `ez_tile_records` from Part 5.1a** (no second split). Reading order: sort by `(y, x)`.

The matplotlib **gallery** uses `gallery_cols_ez` for layout only.


In [ ]:
def ocr_tile_easyocr(tile_path):
    res = reader.readtext(tile_path)
    texts = [r[1] for r in res]
    return '\n'.join(texts).strip(), res


def show_ez_tile_gallery(records, cols=3, title='Part 5.1b — Tile gallery'):
    n = len(records)
    if n == 0:
        print('No tiles to show.')
        return
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows))
    axes = np.atleast_1d(axes).flatten()
    for i, r in enumerate(records):
        im = cv2.imread(r['path'])
        if im is None:
            continue
        axes[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[i].set_title(f"Tile {r['tile_id']}  (row={r.get('row')}, col={r.get('col')})")
        axes[i].axis('off')
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()


gallery_cols_ez = 3
demo_ez = ez_tile_records

print('5.1b | EasyOCR on', len(demo_ez), 'tiles from Part 5.1a')
show_image(clean_path, 'Part 5.1b — Full page (reference)')
show_ez_tile_gallery(demo_ez, cols=gallery_cols_ez)

merged_ez_chunks = []
annotated_ez = []
for r in sorted(demo_ez, key=lambda z: (z['y'], z['x'])):
    txt, raw = ocr_tile_easyocr(r['path'])
    merged_ez_chunks.append(
        f"--- Tile {r['tile_id']} (y={r['y']}, x={r['x']}) ---\n{txt}"
    )
    vis = draw_easyocr_overlay_bgr(r['path'], raw)
    outp = os.path.join('data', f"tile_ez_annot_{r['tile_id']}.png")
    cv2.imwrite(outp, vis)
    annotated_ez.append({**r, 'path': outp})

merged_easyocr_text = '\n\n'.join(merged_ez_chunks)
print('\n' + '=' * 60)
print('MERGED EasyOCR (all tiles, reading order)')
print('=' * 60 + '\n')
print(merged_easyocr_text)

if annotated_ez:
    show_ez_tile_gallery(
        annotated_ez,
        cols=gallery_cols_ez,
        title='Part 5.1b — Tiles with EasyOCR boxes + labels',
    )


### Part 5.1c — Stitch tiles and draw all EasyOCR text on the full image

Rebuild the page from `ez_tile_records`, then run EasyOCR per tile and **offset each polygon** by the tile origin so labels land on the stitched canvas. Saved file: `data/part51c_easyocr_stitched_overlay.png`.


In [ ]:
def draw_easyocr_on_stitched_canvas(canvas_bgr, records):
    out = canvas_bgr.copy()
    for r in sorted(records, key=lambda z: (z['y'], z['x'])):
        results = reader.readtext(r['path'])
        ox, oy = int(r['x']), int(r['y'])
        for bbox, text, conf in results:
            pts = np.array(bbox, dtype=np.int32)
            pts[:, 0] += ox
            pts[:, 1] += oy
            cv2.polylines(out, [pts], True, (0, 200, 255), 2)
            x0, y0 = int(pts[0][0]), int(pts[0][1])
            t = (text or '').strip()
            lab = (t[:40] + '...') if len(t) > 40 else t
            cv2.putText(
                out,
                f'{lab} ({float(conf):.2f})',
                (x0, max(18, y0 - 6)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (200, 0, 120),
                1,
                cv2.LINE_AA,
            )
    return out


stitched_ez = stitch_tiles_to_canvas(ez_tile_records, ez_page_hw)
stitched_ez_ocr = draw_easyocr_on_stitched_canvas(stitched_ez, ez_tile_records)
part51c_path = 'data/part51c_easyocr_stitched_overlay.png'
cv2.imwrite(part51c_path, stitched_ez_ocr)

orig_rgb = cv2.cvtColor(cv2.imread(clean_path), cv2.COLOR_BGR2RGB)
stitch_rgb = cv2.cvtColor(stitched_ez_ocr, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
ax[0].imshow(orig_rgb)
ax[0].set_title('Original page')
ax[0].axis('off')
ax[1].imshow(stitch_rgb)
ax[1].set_title('Stitched + EasyOCR (all tiles mapped to page)')
ax[1].axis('off')
plt.suptitle('Part 5.1c — Original vs stitched with EasyOCR overlay', y=1.02)
plt.tight_layout()
plt.show()

print('Saved:', part51c_path)


### Part 5.2 (Optional) Deep Learning OCR with TrOCR

In this optional section, we focus on **TrOCR** only.

What is TrOCR?
- TrOCR is a transformer-based OCR model from Microsoft.
- It uses a vision encoder + text decoder architecture.
- It is strong for printed text recognition and can generalize well across many document styles.

Why this section is optional:
- It needs extra deep learning dependencies.
- First run downloads model weights, so it can be slow.
- CPU inference is possible, but slower than lightweight OCR engines.

If your environment is limited, you can skip this section and continue with the rest of the notebook.

In [ ]:
# Optional installs for TrOCR
# Uncomment and run if needed:
# %pip install -q transformers torch torchvision sentencepiece

#### Part 5.2 Code — Load and run TrOCR

Run the next code cells to:
1. load TrOCR model/processor,
2. run OCR on `clean_path` and `noisy_path`,
3. optionally test `preprocessed_pdf_path` for comparison.

If dependencies are missing, use the optional install cell above and rerun.

In [ ]:
# Part 5.2 - Step 1) Load TrOCR model and processor once.
# We keep them in memory to avoid reloading for every image.
def load_trocr(model_name='microsoft/trocr-base-printed', suppress_warnings=True):
    import torch
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    from transformers.utils import logging as hf_logging

    if suppress_warnings:
        # Hide non-critical loading warnings (e.g., newly initialized pooler params).
        hf_logging.set_verbosity_error()

    processor = TrOCRProcessor.from_pretrained(model_name)
    model = VisionEncoderDecoderModel.from_pretrained(model_name)
    model.eval()
    device = torch.device('cpu')
    model.to(device)
    return processor, model, device


# Part 5.2 - Step 2) Run OCR inference with TrOCR.
def ocr_trocr(image_path, processor, model, device, max_new_tokens=128):
    """
    TrOCR inference flow:
    1) Read image
    2) Convert image to model input tensor
    3) Generate token ids with decoder
    4) Decode token ids into plain text
    """
    import torch
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(images=image, return_tensors='pt').pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=max_new_tokens,
        )
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text


def draw_trocr_text_on_image(image_path, text, title='TrOCR Output', max_chars_per_line=48):
    import textwrap
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(image_path)
    h, w = img.shape[:2]
    panel_h = max(180, int(h * 0.28))
    canvas = np.full((h + panel_h, w, 3), 255, dtype=np.uint8)
    canvas[:h, :w] = img
    cv2.line(canvas, (0, h), (w, h), (220, 220, 220), 2)
    cv2.putText(canvas, title, (12, h + 28), cv2.FONT_HERSHEY_SIMPLEX, 0.72, (20, 20, 20), 2, cv2.LINE_AA)
    lines = textwrap.wrap((text or '').strip(), width=max_chars_per_line) or ['<empty OCR output>']
    y = h + 56
    for ln in lines:
        if y > h + panel_h - 12:
            cv2.putText(canvas, '...', (12, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (30, 30, 30), 1, cv2.LINE_AA)
            break
        cv2.putText(canvas, ln, (12, y), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (30, 30, 30), 1, cv2.LINE_AA)
        y += 22
    return canvas


def save_trocr_outputs(image_path, text, tag, model_name=None):
    import json
    os.makedirs('data', exist_ok=True)
    txt_path = f'data/part52_trocr_{tag}.txt'
    json_path = f'data/part52_trocr_{tag}.json'
    img_path = f'data/part52_trocr_{tag}_overlay.png'
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write((text or '').strip())
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump({'tag': tag, 'image_path': image_path, 'model_name': model_name, 'text': (text or '').strip()}, f, ensure_ascii=False, indent=2)
    vis = draw_trocr_text_on_image(image_path, text, title=f'TrOCR ({tag})')
    cv2.imwrite(img_path, vis)
    return txt_path, json_path, img_path


try:
    # Try multiple TrOCR checkpoints and keep the first one with non-empty output.
    trocr_candidates = [
        'microsoft/trocr-large-printed',
        'microsoft/trocr-base-printed',
        'microsoft/trocr-small-printed',
        'microsoft/trocr-base-handwritten',
    ]

    trocr_selected_model = None
    trocr_processor = trocr_model = trocr_device = None
    trocr_clean = trocr_noisy = ''

    for model_name in trocr_candidates:
        try:
            print(f'Trying TrOCR model: {model_name}')
            p, m, d = load_trocr(model_name, suppress_warnings=True)
            c = ocr_trocr(clean_path, p, m, d)
            n = ocr_trocr(noisy_path, p, m, d)
            if (c or '').strip() or (n or '').strip():
                trocr_selected_model = model_name
                trocr_processor, trocr_model, trocr_device = p, m, d
                trocr_clean, trocr_noisy = c, n
                break
        except Exception as inner_e:
            print(f'  -> failed: {inner_e}')

    if trocr_selected_model is None:
        raise RuntimeError('All TrOCR candidate models failed or returned empty text.')

    print(f'\nSelected TrOCR model: {trocr_selected_model}')

    print('TrOCR result (clean image):\n')
    print(trocr_clean)

    print('\n' + '='*60 + '\n')
    print('TrOCR result (noisy image):\n')
    print(trocr_noisy)

    clean_txt, clean_json, clean_img = save_trocr_outputs(clean_path, trocr_clean, 'clean', trocr_selected_model)
    noisy_txt, noisy_json, noisy_img = save_trocr_outputs(noisy_path, trocr_noisy, 'noisy', trocr_selected_model)
    show_image(clean_img, 'Part 5.2 — TrOCR clean output (text rendered on image)')
    show_image(noisy_img, 'Part 5.2 — TrOCR noisy output (text rendered on image)')
    print('\nSaved outputs (clean):', clean_txt, clean_json, clean_img)
    print('Saved outputs (noisy):', noisy_txt, noisy_json, noisy_img)
except Exception as e:
    print('TrOCR example skipped or failed.')
    print('Tips:')
    print('- Install optional dependencies in the previous cell.')
    print('- Ensure internet is available for first model download.')
    print('- Retry after restarting kernel if memory is low.')
    print('Error:', e)

#### TrOCR Notes and Tuning Tips

How to interpret output quality:
- If clean image is good but noisy image is poor, add stronger preprocessing before TrOCR.
- If text is truncated, increase `max_new_tokens` in `ocr_trocr(...)`.
- If OCR is slow, keep model loaded once and run multiple images in a loop.

Recommended experiments for learners:
1. Compare TrOCR result on `clean_path`, `noisy_path`, and `prep_path`.
2. Try another checkpoint (for example handwritten-focused models if your input is handwriting).
3. Build a helper that batches multiple page images and stores results to a `.txt` file.

In [ ]:
# Extra comparison for TrOCR with a preprocessed image from Part 3.
# This helps learners see whether preprocessing improves deep OCR.
try:
    trocr_prep = ocr_trocr(preprocessed_pdf_path, trocr_processor, trocr_model, trocr_device)
    print('TrOCR result (preprocessed image):\n')
    print(trocr_prep)
    prep_txt, prep_json, prep_img = save_trocr_outputs(preprocessed_pdf_path, trocr_prep, 'preprocessed', trocr_selected_model)
    show_image(prep_img, 'Part 5.2 — TrOCR preprocessed output (text rendered on image)')
    print('Saved outputs (preprocessed):', prep_txt, prep_json, prep_img)
except Exception as e:
    print('Preprocessed TrOCR run skipped or failed:', e)

## Part 6 - Improve OCR with Preprocessing

Preprocessing is often the biggest practical improvement for OCR quality.

Common methods:
- Grayscale conversion
- Denoising
- Binarization (thresholding)
- Deskewing / de-rotation
- Contrast enhancement

Below we create a simple preprocessing pipeline and test it with Tesseract.

In [ ]:
def preprocess_for_ocr(image_path, save_path='data/preprocessed.png'):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Reduce noise while preserving edges
    denoised = cv2.bilateralFilter(gray, 9, 75, 75)

    # Adaptive threshold for uneven lighting
    bw = cv2.adaptiveThreshold(
        denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 31, 15
    )

    cv2.imwrite(save_path, bw)
    return save_path

prep_path = preprocess_for_ocr(noisy_path)
show_image(prep_path, 'Preprocessed Image for OCR')

print('Tesseract result after preprocessing:\n')
print(ocr_tesseract(prep_path))

## Part 7 - Compare OCR Outputs

This section helps learners inspect OCR behavior in a structured way.

We compare **both clean and noisy images** across OCR methods, then:
1. print extracted text,
2. draw OCR overlays (boxes + labels / rendered text),
3. show side-by-side visual comparisons,
4. save all output images under `data/part7_...`.


In [ ]:
import os

os.makedirs('data', exist_ok=True)

# ---------- 1) Extract text (clean + noisy) ----------
tess_clean = ocr_tesseract(clean_path)
tess_noisy = ocr_tesseract(noisy_path)
easy_clean, easy_clean_raw = ocr_easyocr(clean_path)
easy_noisy, easy_noisy_raw = ocr_easyocr(noisy_path)

# TrOCR may be unavailable on some environments; keep this section optional-safe.
trocr_clean = ''
trocr_noisy = ''
trocr_ok = ('trocr_processor' in globals()) and ('trocr_model' in globals()) and ('trocr_device' in globals())
if trocr_ok:
    try:
        trocr_clean = ocr_trocr(clean_path, trocr_processor, trocr_model, trocr_device)
        trocr_noisy = ocr_trocr(noisy_path, trocr_processor, trocr_model, trocr_device)
    except Exception as e:
        trocr_ok = False
        print('Part 7 note: TrOCR inference failed, skip TrOCR in this section:', e)
else:
    print('Part 7 note: TrOCR model not loaded in memory, skip TrOCR in this section.')

print('--- Tesseract (clean) ---')
print(tess_clean)
print('\n--- Tesseract (noisy) ---')
print(tess_noisy)
print('\n--- EasyOCR (clean) ---')
print(easy_clean)
print('\n--- EasyOCR (noisy) ---')
print(easy_noisy)
if trocr_ok:
    print('\n--- TrOCR (clean) ---')
    print(trocr_clean)
    print('\n--- TrOCR (noisy) ---')
    print(trocr_noisy)

# ---------- 2) Draw overlays and save images ----------
part7_paths = {}

# Tesseract overlays
tess_clean_img = draw_tesseract_word_boxes_on_image(clean_path, lang='eng', psm=6)
tess_noisy_img = draw_tesseract_word_boxes_on_image(noisy_path, lang='eng', psm=6)
part7_paths['tess_clean'] = 'data/part7_tesseract_clean_overlay.png'
part7_paths['tess_noisy'] = 'data/part7_tesseract_noisy_overlay.png'
cv2.imwrite(part7_paths['tess_clean'], tess_clean_img)
cv2.imwrite(part7_paths['tess_noisy'], tess_noisy_img)

# EasyOCR overlays
easy_clean_img = draw_easyocr_overlay_bgr(clean_path, easy_clean_raw)
easy_noisy_img = draw_easyocr_overlay_bgr(noisy_path, easy_noisy_raw)
part7_paths['easy_clean'] = 'data/part7_easyocr_clean_overlay.png'
part7_paths['easy_noisy'] = 'data/part7_easyocr_noisy_overlay.png'
cv2.imwrite(part7_paths['easy_clean'], easy_clean_img)
cv2.imwrite(part7_paths['easy_noisy'], easy_noisy_img)

# TrOCR overlays (rendered text panel)
if trocr_ok:
    trocr_clean_img = draw_trocr_text_on_image(clean_path, trocr_clean, title='Part 7 TrOCR (clean)')
    trocr_noisy_img = draw_trocr_text_on_image(noisy_path, trocr_noisy, title='Part 7 TrOCR (noisy)')
    part7_paths['trocr_clean'] = 'data/part7_trocr_clean_overlay.png'
    part7_paths['trocr_noisy'] = 'data/part7_trocr_noisy_overlay.png'
    cv2.imwrite(part7_paths['trocr_clean'], trocr_clean_img)
    cv2.imwrite(part7_paths['trocr_noisy'], trocr_noisy_img)

# ---------- 3) Show side-by-side image comparisons ----------
def show_compare_row(img_paths, row_title):
    n = len(img_paths)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    axes = np.atleast_1d(axes)
    for i, (title, p) in enumerate(img_paths):
        im = cv2.imread(p)
        axes[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title)
        axes[i].axis('off')
    plt.suptitle(row_title, y=1.02)
    plt.tight_layout()
    plt.show()

clean_row = [
    ('Tesseract (clean)', part7_paths['tess_clean']),
    ('EasyOCR (clean)', part7_paths['easy_clean']),
]
noisy_row = [
    ('Tesseract (noisy)', part7_paths['tess_noisy']),
    ('EasyOCR (noisy)', part7_paths['easy_noisy']),
]
if trocr_ok:
    clean_row.append(('TrOCR (clean)', part7_paths['trocr_clean']))
    noisy_row.append(('TrOCR (noisy)', part7_paths['trocr_noisy']))

show_compare_row(clean_row, 'Part 7 — OCR overlay comparison on clean image')
show_compare_row(noisy_row, 'Part 7 — OCR overlay comparison on noisy image')

print('\nSaved Part 7 overlay images:')
for k, v in part7_paths.items():
    print(f'- {k}: {v}')


## Part 8 - Build a Reusable OCR Pipeline Function

Now we package everything into one helper function for practical usage.

The function supports:
- `method='tesseract'` or `method='easyocr'`
- optional preprocessing (`preprocess=True`)

This is similar to what you might do in a real project.

In [ ]:
def extract_text(image_path, method='tesseract', preprocess=False):
    working_path = image_path
    if preprocess:
        working_path = preprocess_for_ocr(image_path, save_path='data/temp_preprocessed.png')

    if method == 'tesseract':
        return ocr_tesseract(working_path)
    if method == 'easyocr':
        text, _ = ocr_easyocr(working_path)
        return text

    raise ValueError("method must be 'tesseract' or 'easyocr'")

print('Pipeline demo:')
print('\n[Tesseract + preprocessing]\n')
print(extract_text(noisy_path, method='tesseract', preprocess=True))

print('\n[EasyOCR without preprocessing]\n')
print(extract_text(noisy_path, method='easyocr', preprocess=False))

## Part 9 - Cloud OCR API (Free Demo, No Manual API Key Signup)

In this section, we test a cloud OCR service using **OCR.space**.

### Why this option?
- It has a public **demo key** (`helloworld`) for learning.
- You do **not** need to create an account just to try.
- Great for classroom/demo workflows.

### Important notes
- Since this is cloud OCR, internet connection is required.
- The public demo key has strict limits and may fail when overloaded.
- For production, use your own API key and follow provider limits/privacy policy.

In [ ]:
def ocr_cloud_ocrspace(image_path, language='eng', api_key='helloworld'):
    """
    OCR.space endpoint.
    - api_key='helloworld' works for demo/testing only.
    - You can pass your own OCR.space API key for better limits.
    """
    url = 'https://api.ocr.space/parse/image'

    with open(image_path, 'rb') as f:
        files = {'filename': f}
        payload = {
            'apikey': api_key,
            'language': language,
            'isOverlayRequired': False,
            'OCREngine': 2
        }
        response = requests.post(url, files=files, data=payload, timeout=60)

    response.raise_for_status()
    data = response.json()

    parsed_text = ''
    if data.get('ParsedResults'):
        parsed_text = data['ParsedResults'][0].get('ParsedText', '')

    return parsed_text, data

# A) Free demo example (no manual API-key signup)
try:
    cloud_text, cloud_raw = ocr_cloud_ocrspace(noisy_path)
    print('Cloud OCR (OCR.space demo key) result:\n')
    print(cloud_text)
except Exception as e:
    print('Cloud OCR request failed.')
    print('Possible reasons: no internet, demo limit reached, temporary API issue.')
    print('Error:', e)

### Part 9.1 (Optional) Cloud OCR APIs with Your Own API Key

This subsection is **optional**.

If you do not have API keys, skip all cells in Part 9.1 and continue to the next section.

Why split these examples into separate cells?
- Each provider has different endpoint, auth style, and response schema.
- Learners can test one provider at a time.
- Errors are isolated and easier to debug.

Best practice:
- Do not hardcode API keys in the notebook.
- Load keys from environment variables.

In [ ]:
# Shared optional setup for API-key providers
MISTRAL_API_KEY = os.getenv('MISTRAL_API_KEY')
GOOGLE_VISION_API_KEY = os.getenv('GOOGLE_VISION_API_KEY')
AZURE_DOCINTEL_ENDPOINT = os.getenv('AZURE_DOCINTEL_ENDPOINT')
AZURE_DOCINTEL_KEY = os.getenv('AZURE_DOCINTEL_KEY')
OCRSPACE_API_KEY = os.getenv('OCRSPACE_API_KEY')

print('Optional provider key availability:')
print('- MISTRAL_API_KEY:', bool(MISTRAL_API_KEY))
print('- GOOGLE_VISION_API_KEY:', bool(GOOGLE_VISION_API_KEY))
print('- AZURE_DOCINTEL_ENDPOINT:', bool(AZURE_DOCINTEL_ENDPOINT))
print('- AZURE_DOCINTEL_KEY:', bool(AZURE_DOCINTEL_KEY))
print('- OCRSPACE_API_KEY:', bool(OCRSPACE_API_KEY))

print('\nIf all are False, skip Part 9.1.')

#### Optional Provider Example 1 - Mistral OCR

Use this when you have `MISTRAL_API_KEY`.

Notes:
- API versions and payload schema can change over time.
- Always verify with official provider documentation before production use.
- This cell prints status and a short response preview for quick debugging.

In [ ]:
def ocr_mistral_example(image_path, api_key):
    url = 'https://api.mistral.ai/v1/ocr'
    headers = {'Authorization': f'Bearer {api_key}'}

    with open(image_path, 'rb') as f:
        files = {'file': f}
        data = {'model': 'mistral-ocr-latest'}
        response = requests.post(url, headers=headers, files=files, data=data, timeout=60)

    return response.status_code, response.text[:500]

if not MISTRAL_API_KEY:
    print('Skip: MISTRAL_API_KEY not found.')
else:
    code, preview = ocr_mistral_example(noisy_path, MISTRAL_API_KEY)
    print('[Mistral OCR] status:', code)
    print(preview)

#### Optional Provider Example 2 - Google Cloud Vision API

Use this when you have `GOOGLE_VISION_API_KEY`.

Notes:
- This example uses `TEXT_DETECTION` on base64-encoded image content.
- The response is JSON and can be large; we only print a short preview.
- In real projects, parse `textAnnotations` for structured output.

In [ ]:
def ocr_google_vision_example(image_path, api_key):
    import base64

    with open(image_path, 'rb') as f:
        content_b64 = base64.b64encode(f.read()).decode('utf-8')

    url = f'https://vision.googleapis.com/v1/images:annotate?key={api_key}'
    payload = {
        'requests': [
            {
                'image': {'content': content_b64},
                'features': [{'type': 'TEXT_DETECTION'}]
            }
        ]
    }

    response = requests.post(url, json=payload, timeout=60)
    return response.status_code, response.text[:500]

if not GOOGLE_VISION_API_KEY:
    print('Skip: GOOGLE_VISION_API_KEY not found.')
else:
    code, preview = ocr_google_vision_example(noisy_path, GOOGLE_VISION_API_KEY)
    print('[Google Vision] status:', code)
    print(preview)

#### Optional Provider Example 3 - Azure AI Document Intelligence

Use this when you have both:
- `AZURE_DOCINTEL_ENDPOINT`
- `AZURE_DOCINTEL_KEY`

Notes:
- This API often returns `202 Accepted` first.
- You usually need a second polling request (using `operation-location`) to fetch final OCR result.
- In this learning cell, we print status and headers only.

In [ ]:
def ocr_azure_docintel_example(image_path, endpoint, api_key):
    url = f"{endpoint}/formrecognizer/documentModels/prebuilt-read:analyze?api-version=2023-07-31"
    headers = {
        'Ocp-Apim-Subscription-Key': api_key,
        'Content-Type': 'application/octet-stream'
    }

    with open(image_path, 'rb') as f:
        response = requests.post(url, headers=headers, data=f.read(), timeout=60)

    return response.status_code, dict(response.headers)

if not (AZURE_DOCINTEL_ENDPOINT and AZURE_DOCINTEL_KEY):
    print('Skip: AZURE_DOCINTEL_ENDPOINT and/or AZURE_DOCINTEL_KEY not found.')
else:
    code, headers = ocr_azure_docintel_example(noisy_path, AZURE_DOCINTEL_ENDPOINT, AZURE_DOCINTEL_KEY)
    print('[Azure Document Intelligence] status:', code)
    print('Response headers:', headers)

## Final Notes and Practical Tips

### When to choose Tesseract
- You need a lightweight, fast, local OCR engine.
- Your documents are mostly clean scans or printed text.

### When to choose EasyOCR
- Your images contain noise, rotation, or complex layouts.
- You want strong baseline performance with less manual tuning.

### When to try Cloud OCR APIs
- You want quick setup without local OCR executable dependencies.
- You need scalable remote processing.
- You can accept internet dependency and sending image data to an external service.

### Typical OCR project workflow
1. Collect representative image samples.
2. Benchmark local OCR and cloud OCR on the same dataset.
3. Add preprocessing based on failure patterns.
4. Evaluate with metrics (character error rate / word accuracy).
5. Wrap into a robust pipeline with error handling.

## Exercises for learners
1. Compare OCR quality between Tesseract, EasyOCR, and OCR.space demo.
2. Try different `--psm` values in Tesseract and compare quality.
3. Add spelling correction post-processing.
4. Evaluate OCR quality on handwritten text.
5. Build a function that returns text + confidence + method name.

Great job! You now have a complete end-to-end OCR learning notebook.

In [ ]:
if not OCRSPACE_API_KEY:
    print('Skip: OCRSPACE_API_KEY not found.')
else:
    try:
        paid_text, _ = ocr_cloud_ocrspace(noisy_path, api_key=OCRSPACE_API_KEY)
        print('[OCR.space personal key] result preview:\n')
        print(paid_text[:1000])
    except Exception as e:
        print('OCR.space personal key example failed:', e)

In [ ]:
# Optional: Compare local vs cloud quickly
print('--- Local: Tesseract (preprocessed) ---')
print(extract_text(noisy_path, method='tesseract', preprocess=True))

print('\n--- Local: EasyOCR ---')
print(extract_text(noisy_path, method='easyocr', preprocess=False))

print('\n--- Cloud: OCR.space (demo key) ---')
try:
    cloud_text, _ = ocr_cloud_ocrspace(noisy_path)
    print(cloud_text)
except Exception as e:
    print('Cloud OCR unavailable right now:', e)